# 02 — Subset: single-point correlation

A `Subset` is a small patch of pixels centred on one point in the reference image.
The DIC solver finds the **warp parameters** that best deform this patch to match the
same region in the target image — giving you the local displacement and strain at that point.

This notebook covers: warp parameters, `solve_icgn` vs `solve_fagn`,
correlation scores, convergence history, `save` / `load`.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import geopyv_dev as gp

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "animation.embed_limit": 50,
})


In [ ]:
# Generate a reference and a deformed speckle image.
# 5 px x-translation + slight shear — fast enough to run in < 2 s.
os.makedirs("data/single", exist_ok=True)
speckle = gp.Speckle(
    image_cfg={"image_dir": "data/single", "name": "demo", "image_size": (401, 401)},
    speckle_cfg={"speckle_size": 3.5, "speckle_number": 700},
    progression="deformation",
    deformation_cfg={"comp": [5.0, 0.0, 0.005, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
    noise_cfg=(0.0, 1.5),
    scale_cfg={"scale": "lin", "n": 2},
)
speckle.solve(seed=42)
f_img = gp.Image(filepath="data/single/demo_0.jpg")
g_img = gp.Image(filepath="data/single/demo_1.jpg")
print(f"Reference : {f_img}")
print(f"Target    : {g_img}")

boundary_nodes = np.array([
    [50.0, 50.0], [50.0, 350.0], [350.0, 350.0], [350.0, 50.0]
])
boundary  = gp.PathRegion(nodes=boundary_nodes, hard=False)
local_mask = gp.Mask(mask_type="local", shape="circle", size=25)
seed_coord = [200.0, 200.0]


## The warp function

The DIC solver describes how a subset deforms using a **warp vector** `p`.

For **order 1** (affine, 6 parameters):

```
u(x,y) = p[0] + p[1]·Δx + p[2]·Δy
v(x,y) = p[3] + p[4]·Δx + p[5]·Δy
```

where Δx = x − x₀, Δy = y − y₀ (offsets from the subset centre x₀, y₀).

For **order 2** (quadratic, 12 parameters), second-order terms p[3..5] and p[9..11]
are added.  Most analyses use order 1 for `Subset` and order 2 for `Mesh`/`Sequence`.

The diagram below labels each parameter.


In [ ]:
# Warp parameter diagram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), facecolor="white")
for ax in (ax1, ax2):
    ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.5); ax.set_aspect("equal"); ax.axis("off")

# Reference square
sq = patches.Rectangle((0.5, 0.5), 2, 2, lw=2, ec="steelblue", fc="steelblue", alpha=0.12)
ax1.add_patch(sq)
ax1.text(1.5, 1.5, "$f$ subset\n(reference)", ha="center", va="center",
         fontsize=10, color="steelblue", fontweight="bold")
ax1.set_title("Reference subset", fontsize=11, fontweight="bold")
ax1.plot(1.5, 1.5, "r+", ms=14, mew=2)
ax1.text(1.6, 1.6, "$(x_0, y_0)$", fontsize=9, color="red")

# Deformed (translate + slight shear)
from matplotlib.patches import FancyArrowPatch
pts = np.array([[0.5, 0.5], [2.5, 0.5], [2.7, 2.5], [0.5, 2.5], [0.5, 0.5]])
pts[:, 0] += 0.3   # u displacement
pts[:, 1] -= 0.25  # v displacement
ax2.add_patch(patches.Polygon(pts[:-1], closed=True, lw=2, ec="seagreen",
                               fc="seagreen", alpha=0.12))
ax2.set_title("Deformed subset (in target $g$)", fontsize=11, fontweight="bold")

# Arrows for u, v
ax2.annotate("", xy=(1.5+0.3, 1.5-0.25), xytext=(1.5, 1.5),
              arrowprops=dict(arrowstyle="-|>", color="crimson", lw=2))
ax2.text(1.75, 1.25, "$u = p_0$", color="crimson", fontsize=10, fontweight="bold")
ax2.annotate("", xy=(1.5+0.3, 1.5-0.25-0.5), xytext=(1.5+0.3, 1.5-0.25),
              arrowprops=dict(arrowstyle="-|>", color="royalblue", lw=2))
ax2.text(1.85, 0.9, "$v = p_3$", color="royalblue", fontsize=10, fontweight="bold")

ax2.text(0.1, 3.2,
         r"$u(\Delta x,\Delta y)=p_0+p_1\Delta x+p_2\Delta y$",
         fontsize=9, color="#333")
ax2.text(0.1, 2.95,
         r"$v(\Delta x,\Delta y)=p_3+p_4\Delta x+p_5\Delta y$",
         fontsize=9, color="#333")

plt.suptitle("Order-1 warp: 6 parameters", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
os.makedirs("../docs/assets", exist_ok=True)
fig.savefig("../docs/assets/nb2_warp_diagram.png", dpi=120, bbox_inches="tight")
plt.show()


## Creating and solving a `Subset`

```python
subset = gp.Subset(
    coord      = [x, y],       # centre of the subset in the reference image
    local_mask = mask,          # a local Mask object defining the patch shape
    f_img      = f_img,         # reference Image
    g_img      = g_img,         # target Image
    subset_order = 1,           # 1 (affine) or 2 (quadratic)
)
subset.solve()                  # runs ICGN by default
```

`solve()` is non-destructive to the images — you can create many subsets from the same pair.


In [ ]:
# Create a subset at the image centre
subset = gp.Subset(
    coord      = [200.0, 200.0],
    local_mask = local_mask,
    f_img      = f_img,
    g_img      = g_img,
    subset_order = 1,
)
print(subset)
print(f"\n  n_px (active pixels) : {subset.n_px}")
print(f"  sssig (intensity STD) : {subset.sssig:.2f}  (> 30 is healthy for DIC)")
print(f"  solved               : {subset.solved}")


In [ ]:
# Inspect the reference patch before solving
subset.inspect()


In [ ]:
# Solve with ICGN (Inverse Compositional Gauss-Newton) — the standard algorithm
subset.solve(algorithm="icgn", max_norm=1e-3, max_iterations=50, tolerance=0.75)
print(subset)
print(f"\n  p      : {[f'{v:.5f}' for v in subset.p]}")
print(f"  u      : {subset.p[0]:.4f} px   (x-displacement)")
print(f"  v      : {subset.p[3]:.4f} px   (y-displacement)")
print(f"  C_ZNCC : {subset.c_zncc:.6f}  (1.0 = perfect match)")
print(f"  iters  : {subset.iterations}")
print(f"  converged : {subset.converged}")


### ICGN vs FAGN

`geopyv_dev` implements two DIC algorithms:

| Algorithm | Full name | Notes |
|-----------|-----------|-------|
| `"icgn"` | Inverse Compositional Gauss-Newton | Faster — Hessian computed once |
| `"fagn"` | Forward Additive Gauss-Newton | Slower — Hessian rebuilt each iteration |

Both converge to the same answer for well-textured subsets.  FAGN can sometimes
recover from poor initial guesses that ICGN struggles with.


In [ ]:
# Convergence animation — replay the iteration history
# history = list of (iteration_no, C_ZNCC, C_ZNSSD, delta_norm)
history = subset.history
iters   = [h[0] for h in history]
zncc    = [h[1] for h in history]
norms   = [h[3] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.set_xlim(-0.5, max(iters) + 0.5); ax1.set_ylim(min(zncc)*0.999, 1.0005)
ax2.set_xlim(-0.5, max(iters) + 0.5); ax2.set_ylim(0, max(norms)*1.15)

for ax, ylabel in [(ax1, "C_ZNCC"), (ax2, "Δ norm (convergence)")]:
    ax.set_xlabel("Iteration"); ax.set_ylabel(ylabel)
    ax.axhline(0 if ax is ax2 else 1.0, ls=":", c="grey", alpha=0.5)

line1, = ax1.plot([], [], "o-", c="steelblue", ms=5, label="C_ZNCC")
line2, = ax2.plot([], [], "o-", c="crimson",   ms=5, label="Δ norm")
dot1   = ax1.plot([], [], "o", c="steelblue", ms=10)[0]
dot2   = ax2.plot([], [], "o", c="crimson",   ms=10)[0]
ax1.axhline(0.75, ls="--", c="orange", lw=1.2, label="tolerance=0.75")
ax1.legend(fontsize=9); ax2.legend(fontsize=9)

def update(frame):
    i = frame + 1
    line1.set_data(iters[:i], zncc[:i]);  dot1.set_data([iters[i-1]], [zncc[i-1]])
    line2.set_data(iters[:i], norms[:i]); dot2.set_data([iters[i-1]], [norms[i-1]])
    return line1, line2, dot1, dot2

anim = FuncAnimation(fig, update, frames=len(history), interval=250, blit=True)
plt.suptitle("ICGN convergence history", fontsize=12, fontweight="bold")
plt.tight_layout()
anim.save("../docs/assets/nb2_convergence.gif", writer="pillow", fps=5)
HTML(anim.to_jshtml())


In [ ]:
# Compare ICGN and FAGN on the same subset
subset_fagn = gp.Subset(coord=[200.0, 200.0], local_mask=local_mask,
                         f_img=f_img, g_img=g_img)
subset_fagn.solve(algorithm="fagn", max_norm=1e-3, max_iterations=50)

print(f"ICGN: p[0]={subset.p[0]:.5f}  iters={subset.iterations}  C_ZNCC={subset.c_zncc:.6f}")
print(f"FAGN: p[0]={subset_fagn.p[0]:.5f}  iters={subset_fagn.iterations}  C_ZNCC={subset_fagn.c_zncc:.6f}")


In [ ]:
# Save to disk and reload
subset.save("data/subset.pyv")

loaded = gp.load("data/subset.pyv")
print(f"Loaded: {loaded}")
print(f"  p[0]   : {loaded.p[0]:.5f}")
print(f"  C_ZNCC : {loaded.c_zncc:.6f}")


## Key takeaways

- `p[0]` = u-displacement, `p[3]` = v-displacement (order-1).
- `C_ZNCC` close to 1.0 means a reliable match; below ~0.7 is suspect.
- `converged` is `True` when the Δ norm dropped below `max_norm` within `max_iterations`.
- ICGN ≈ 5–10× faster than FAGN for well-conditioned problems.

**Next:** Notebook 03 — extending to full-field measurements with `Mesh`.
